MONITORING + DRIFT DETECTION

A model isn't finished when you deploy it.

Suppose six months later:

Transaction patterns change
        ↓
Feature distribution changes
        ↓
Model confidence changes
        ↓
Fraud detection degrades

We need monitoring.

We'll compare train vs validation distributions.

In [95]:
# ================================================================
# STAGE 15 — MODEL MONITORING + DRIFT
# ================================================================

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

print("=" * 70)
print("STAGE 15 — MODEL MONITORING + DRIFT DETECTION")
print("=" * 70)

drift_results = []

for column in X_train.columns:

    train_values = X_train[column].dropna()
    current_values = X_val[column].dropna()

    statistic, p_value = ks_2samp(
        train_values,
        current_values
    )

    drift_results.append({
        "Feature": column,
        "KS_Statistic": statistic,
        "P_Value": p_value
    })

drift_df = pd.DataFrame(
    drift_results
)

drift_df = drift_df.sort_values(
    "KS_Statistic",
    ascending=False
)

print("\nTop potential distribution shifts:")
display(
    drift_df.head(15)
)

drift_df.to_csv(
    "stage15_drift_report.csv",
    index=False
)

print(
    "\nSaved: stage15_drift_report.csv"
)

STAGE 15 — MODEL MONITORING + DRIFT DETECTION

Top potential distribution shifts:


,Feature,KS_Statistic,P_Value
21,V21,0.009988,0.012967
19,V19,0.009546,0.020054
11,V11,0.009148,0.029173
29,Amount,0.007983,0.079920
28,V28,0.007931,0.083307
23,V23,0.007564,0.111048
3,V3,0.006286,0.270746
13,V13,0.006162,0.292559
25,V25,0.006125,0.299159
1,V1,0.006060,0.311222



Saved: stage15_drift_report.csv


In [96]:
# ================================================================
# FINAL EVALUATION — TEST SET
# ================================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

# ------------------------------------------------
# IMPORTANT:
# Use the exact feature representation used by
# the selected model.
# ------------------------------------------------

test_proba = best_xgb.predict_proba(
    X_test
)[:, 1]

# ------------------------------------------------
# Frozen threshold
# ------------------------------------------------

FINAL_THRESHOLD = 0.18

test_pred = (
    test_proba >= FINAL_THRESHOLD
).astype(int)

# ------------------------------------------------
# Metrics
# ------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred
).ravel()

precision = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else 0
)

recall = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

roc_auc = roc_auc_score(
    y_test,
    test_proba
)

pr_auc = average_precision_score(
    y_test,
    test_proba
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

print("\nTP :", tp)
print("FP :", fp)
print("FN :", fn)
print("TN :", tn)

print("\nPrecision :", precision)
print("Recall    :", recall)
print("F1        :", f1)
print("ROC-AUC   :", roc_auc)
print("PR-AUC    :", pr_auc)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        digits=4
    )
)

FINAL TEST EVALUATION

Confusion Matrix:
[[28324     2]
 [    9    38]]

TP : 38
FP : 2
FN : 9
TN : 28324

Precision : 0.95
Recall    : 0.8085106382978723
F1        : 0.8735632183908046
ROC-AUC   : 0.9858741912174516
PR-AUC    : 0.8476260715655435

Classification Report:
              precision    recall  f1-score   support

           0     0.9997    0.9999    0.9998     28326
           1     0.9500    0.8085    0.8736        47

    accuracy                         0.9996     28373
   macro avg     0.9748    0.9042    0.9367     28373
weighted avg     0.9996    0.9996    0.9996     28373



In [97]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Change this folder name if you want
DRIVE_FOLDER = "/content/drive/MyDrive/Fraud_Detection_Project"

os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Copy everything from /content into the Drive folder
for item in os.listdir("/content"):
    src = os.path.join("/content", item)

    # Don't copy the Drive mount into itself
    if item == "drive":
        continue

    dst = os.path.join(DRIVE_FOLDER, item)

    try:
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.copytree(src, dst, dirs_exist_ok=True)
            else:
                shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    except Exception as e:
        print(f"Skipped {item}: {e}")

print("✅ Everything saved to:")
print(DRIVE_FOLDER)

Mounted at /content/drive
✅ Everything saved to:
/content/drive/MyDrive/Fraud_Detection_Project


In [98]:
!du -sh "/content/drive/MyDrive/Fraud_Detection_Project"
!find "/content/drive/MyDrive/Fraud_Detection_Project" -maxdepth 2 -type f | head -50

341M	/content/drive/MyDrive/Fraud_Detection_Project
/content/drive/MyDrive/Fraud_Detection_Project/.config/config_sentinel
/content/drive/MyDrive/Fraud_Detection_Project/.config/gce
/content/drive/MyDrive/Fraud_Detection_Project/.config/.last_opt_in_prompt.yaml
/content/drive/MyDrive/Fraud_Detection_Project/.config/.last_survey_prompt.yaml
/content/drive/MyDrive/Fraud_Detection_Project/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/drive/MyDrive/Fraud_Detection_Project/.config/active_config
/content/drive/MyDrive/Fraud_Detection_Project/.config/default_configs.db
/content/drive/MyDrive/Fraud_Detection_Project/.config/.last_update_check.json
/content/drive/MyDrive/Fraud_Detection_Project/stage12_ensemble_results.csv
/content/drive/MyDrive/Fraud_Detection_Project/fraud_xgboost_model.pkl
/content/drive/MyDrive/Fraud_Detection_Project/fraud_scaler.pkl
/content/drive/MyDrive/Fraud_Detection_Project/creditcardfraud.zip
/content/drive/MyDrive/Fraud_Detection_P